<a href="https://colab.research.google.com/github/Hansini23/Statistical-Learning-e23291/blob/main/ME2050_Assignment_07c.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E/23/291

# Bayesian Inference — Assignment Solutions



---
# Question 1 — Bayesian Estimation of a User Ability Parameter from Item Responses

**Setup.** A learner answers items $1,\dots,n$ one at a time. Response $Y_i \in \{0,1\}$ given ability $\theta$ follows the 2PL model
$$p_i(\theta) = P(Y_i = 1 \mid \Theta=\theta) = \frac{1}{1+e^{-a_i(\theta - b_i)}}.$$
The prior is $\Theta \sim \mathscr N(0,1)$, and after each response the posterior becomes the prior for the next item.

## Task 1 — Visualizing the 2PL Curve

The parameter $a_i$ controls how *steeply* the probability of a correct answer rises with ability (the slope at the inflection point), while $b_i$ marks *where* that inflection point sits on the ability scale — it is the ability level at which $p_i(\theta) = 0.5$.

Below, two discrimination values are compared ($a=0.8$, a fairly "soft" item, and $a=1.8$, a "sharp" item), and the sharp item is shown at three difficulty settings $b \in \{-1, 0, 1\}$.

In [1]:
import numpy as np
import plotly.graph_objects as go

def irt_prob(theta, a, b):
    """2PL probability of a correct response."""
    return 1.0 / (1.0 + np.exp(-a * (theta - b)))

theta_axis = np.linspace(-6, 6, 400)

item_settings = [
    dict(a=0.8, b=0.0,  label="a=0.8, b=0 (low discrimination)", dash="dot",   color="#888888"),
    dict(a=1.8, b=-1.0, label="a=1.8, b=-1 (easy)",               dash="solid", color="#1f77b4"),
    dict(a=1.8, b=0.0,  label="a=1.8, b=0 (medium)",              dash="solid", color="#d62728"),
    dict(a=1.8, b=1.0,  label="a=1.8, b=1 (hard)",                dash="solid", color="#2ca02c"),
]

fig = go.Figure()
for cfg in item_settings:
    fig.add_trace(go.Scatter(
        x=theta_axis, y=irt_prob(theta_axis, cfg["a"], cfg["b"]),
        mode="lines", name=cfg["label"],
        line=dict(color=cfg["color"], dash=cfg["dash"], width=3)
    ))

fig.update_layout(
    title="2PL Item Characteristic Curves",
    xaxis_title="Ability θ",
    yaxis_title="P(correct | θ)",
    template="plotly_white",
    legend=dict(x=0.02, y=0.98)
)
fig.show()

**Reading the plot:** all three $a=1.8$ curves are identical in shape, just slid left/right — raising $b$ pushes the curve to the right, meaning a *higher* ability is needed to have a 50% chance of getting the item correct (the item is harder). The dashed $a=0.8$ curve rises much more gradually, so it is less able to separate low- from high-ability users; the steep $a=1.8$ curves discriminate sharply around their midpoint.

## Task 2 — Likelihood of One Response and the Running History

A single response $y_k$ is a Bernoulli outcome with success probability $p_k(\theta)$, so its likelihood contribution is
$$L(y_k \mid \theta) = p_k(\theta)^{\,y_k}\,\big(1-p_k(\theta)\big)^{\,1-y_k}.$$

Assuming responses are conditionally independent given $\theta$ (a standard IRT assumption), the likelihood of the entire history observed so far, $\mathbf y^{(k)}=(y_1,\dots,y_k)$, is the product of the individual terms:
$$L(\mathbf y^{(k)} \mid \theta) = \prod_{i=1}^{k} p_i(\theta)^{\,y_i}\big(1-p_i(\theta)\big)^{\,1-y_i}.$$

## Task 3 — Recursive Posterior Update

Because the previous posterior becomes the current prior, Bayes' rule applied at step $k$ gives
$$f_{\Theta\mid\mathbf Y^{(k)}}(\theta\mid\mathbf y^{(k)}) \;\propto\; L(y_k\mid\theta)\; f_{\Theta\mid\mathbf Y^{(k-1)}}(\theta\mid\mathbf y^{(k-1)}),$$
with the constant of proportionality being $1/\int L(y_k\mid s)\,f_{\Theta\mid\mathbf Y^{(k-1)}}(s\mid\mathbf y^{(k-1)})\,ds$. Starting condition: $f_{\Theta\mid\mathbf Y^{(0)}} = f_\Theta^{(0)} = \mathscr N(0,1)$ density.

## Task 4 — Effect of a Correct Answer on a Hard Item

If $y_k=1$ and $b_k$ is large (a difficult item), the likelihood factor $p_k(\theta)$ is close to $0$ for small/moderate $\theta$ and only rises to values near $1$ for $\theta$ well above $b_k$. Multiplying the previous posterior by this factor suppresses the left tail of the distribution and amplifies the right tail — the probability mass, and hence the peak, is pulled *to the right*, toward higher ability. The size of the shift is larger the harder the item was, because getting a hard item right is stronger evidence of high ability than getting an easy item right.

## Task 5 — Effect of Discrimination on Sharpness

The steepness of $p_k(\theta)$ near $\theta=b_k$ is governed by $a_k$. A large $a_k$ makes the likelihood function change quickly from near 0 to near 1 as $\theta$ crosses $b_k$ — it behaves almost like a step function, so multiplying it into the posterior removes probability mass aggressively on one side, producing a narrow, confident (low-variance) posterior. A small $a_k$ changes very gradually with $\theta$, so it barely reshapes the posterior — the update is weak and the resulting posterior stays almost as spread out as before. In the limit $a_k\to 0$, the item carries essentially no information about $\theta$.

## Task 6 — Numerical Grid Algorithm

1. Fix a dense grid $\theta_1 < \theta_2 < \dots < \theta_M$ spanning the plausible ability range (e.g. $[-5,5]$) and evaluate the initial $\mathscr N(0,1)$ density on it; call this array `belief`.
2. When item $k$ (with known $a_k, b_k$) is answered with outcome $y_k$, evaluate the item's response probability on the whole grid, form the pointwise likelihood array `like = p**y_k * (1-p)**(1-y_k)`, and multiply it elementwise into `belief`.
3. Renormalize so the array integrates to 1 over the grid using the composite trapezoidal rule: `Z = np.trapezoid(belief, theta_grid)` then `belief /= Z`. This is the sequential normalization step — it replaces the analytic integral in the denominator of Bayes' rule with a numerical one.
4. Repeat step 2–3 for each new item; `belief` after step $k$ *is* the discretized posterior $f_{\Theta\mid\mathbf Y^{(k)}}$, ready to serve as the prior for step $k+1$.
5. Point estimates are read off the grid: posterior mean via `np.trapezoid(theta_grid*belief, theta_grid)`, MAP via `theta_grid[np.argmax(belief)]`.

## Task 7 — Convergence Simulation over 20 Items

In [2]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

rng = np.random.default_rng(7)

true_ability = 0.75
n_items = 20
theta_grid = np.linspace(-5, 5, 800)

belief = stats.norm.pdf(theta_grid, 0, 1)   # initial prior

bayes_track = [np.trapezoid(theta_grid * belief, theta_grid)]
map_track   = [theta_grid[np.argmax(belief)]]

for k in range(n_items):
    a_k = rng.uniform(0.5, 2.0)
    b_k = rng.normal(0, 1)

    true_p = irt_prob(true_ability, a_k, b_k)
    y_k = 1 if rng.uniform() < true_p else 0

    p_grid = irt_prob(theta_grid, a_k, b_k)
    like = p_grid**y_k * (1 - p_grid)**(1 - y_k)

    belief = belief * like
    belief /= np.trapezoid(belief, theta_grid)

    bayes_track.append(np.trapezoid(theta_grid * belief, theta_grid))
    map_track.append(theta_grid[np.argmax(belief)])

steps = list(range(n_items + 1))

fig = go.Figure()
fig.add_hline(y=true_ability, line_dash="dash", line_color="crimson",
              annotation_text=f"true θ = {true_ability}")
fig.add_trace(go.Scatter(x=steps, y=bayes_track, mode="lines+markers",
                          name="Posterior mean estimate", line=dict(color="#1f77b4", width=2.5)))
fig.add_trace(go.Scatter(x=steps, y=map_track, mode="lines+markers",
                          name="MAP estimate", marker_symbol="square",
                          line=dict(color="#2ca02c", width=2, dash="dot")))
fig.update_layout(
    title="Ability Estimate Convergence over 20 Items",
    xaxis_title="Item number k", yaxis_title="Estimated ability",
    template="plotly_white", hovermode="x unified"
)
fig.show()

print("Final posterior mean:", round(bayes_track[-1], 3))
print("Final MAP estimate:  ", round(map_track[-1], 3))

Final posterior mean: 0.911
Final MAP estimate:   0.895


**Analysis:** both estimators start far from $\theta_{\text{true}}=0.75$ (the prior mean is 0), and the gap shrinks as more items accumulate, though not monotonically — a single unlucky response can pull an estimate briefly away from the truth. By item 15–20 the two estimators typically sit within a small band around 0.75 and stay close together, since with enough evidence the posterior becomes unimodal and fairly symmetric, making the mean and mode nearly coincide. This reflects growing confidence: as evidence accumulates the posterior concentrates, so its mean and mode become increasingly reliable point summaries of the learner's true ability.

---
# Question 2 — Bayesian Tracking of Click-Trough Rates via Conjugate Beta-Binomial Updates

**Setup.** Each impression $k$ produces a click indicator $Y_k\in\{0,1\}$ with $P(Y_k=1\mid\Theta=\theta)=\theta$, and the belief about $\theta$ starts at $\Theta\sim\text{Beta}(\alpha_0,\beta_0)$, updated sequentially as impressions arrive.

## Task 1 — Shapes of the Beta Prior

$\alpha$ and $\beta$ act like pseudo-counts of prior "successes" and "failures": the distribution's mass shifts toward 1 as $\alpha$ grows relative to $\beta$, and toward 0 as $\beta$ dominates. When $\alpha=\beta$ the density is symmetric about 0.5 (and flat/uniform when both equal 1).

In [3]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

theta_axis = np.linspace(0, 1, 400)

beta_settings = [
    dict(a=1, b=1, label="Beta(1,1) — flat, uninformative", color="#888888", dash="dot"),
    dict(a=2, b=8, label="Beta(2,8) — mass near 0 (low CTR belief)", color="#1f77b4", dash="solid"),
    dict(a=8, b=2, label="Beta(8,2) — mass near 1 (high CTR belief)", color="#d62728", dash="solid"),
]

fig = go.Figure()
for cfg in beta_settings:
    fig.add_trace(go.Scatter(
        x=theta_axis, y=stats.beta.pdf(theta_axis, cfg["a"], cfg["b"]),
        mode="lines", name=cfg["label"],
        line=dict(color=cfg["color"], dash=cfg["dash"], width=3)
    ))

fig.update_layout(
    title="Beta(α, β) Density Shapes",
    xaxis_title="θ (click-through rate)", yaxis_title="Density",
    template="plotly_white"
)
fig.show()

## Task 2 — Single-Trial and Joint Likelihood

Each impression is Bernoulli($\theta$), so a single observation contributes
$$L(y_k\mid\theta)=\theta^{y_k}(1-\theta)^{1-y_k}.$$
Assuming impressions are conditionally independent given $\theta$, the joint likelihood for the history $\mathbf y^{(k)}$ is
$$L(\mathbf y^{(k)}\mid\theta)=\prod_{i=1}^k \theta^{y_i}(1-\theta)^{1-y_i} = \theta^{\,c_k}(1-\theta)^{\,k-c_k},$$
where $c_k=\sum_{i=1}^k y_i$ is the running click count.

## Task 3 — Closed-Form Conjugate Update

Write the prior inherited from step $k-1$ as $\text{Beta}(\alpha_{k-1},\beta_{k-1})$, so its density is proportional to $\theta^{\alpha_{k-1}-1}(1-\theta)^{\beta_{k-1}-1}$. Multiplying by the single-trial likelihood $\theta^{y_k}(1-\theta)^{1-y_k}$ and combining exponents gives
$$f_{\Theta\mid\mathbf Y^{(k)}}(\theta\mid\mathbf y^{(k)}) \;\propto\; \theta^{\,(\alpha_{k-1}+y_k)-1}(1-\theta)^{\,(\beta_{k-1}+1-y_k)-1}.$$
This kernel is exactly a Beta density, so the posterior stays in the Beta family (conjugacy), with updated parameters
$$\alpha_k=\alpha_{k-1}+y_k, \qquad \beta_k=\beta_{k-1}+(1-y_k).$$
The posterior mean at step $k$ follows immediately from the Beta mean formula:
$$\mathbb E[\Theta\mid\mathbf Y^{(k)}=\mathbf y^{(k)}] = \frac{\alpha_k}{\alpha_k+\beta_k}.$$

## Task 4 — Click vs. No-Click Shift, and Contrast with the Non-Conjugate Case

A click ($y_k=1$) increments $\alpha$, sliding the whole distribution — and its peak — toward 1; a non-click ($y_k=0$) increments $\beta$, sliding it toward 0. Each update is a single arithmetic increment, so the entire posterior shape is available in closed form at every step — no numerical integration is ever needed. This is the key contrast with the 2PL ability model of Question 1: there, the logistic likelihood is *not* conjugate to a Normal prior, so the posterior has no closed-form family and must be tracked with a discretized grid and repeated trapezoidal normalization at every step.

## Task 5 — Closed-Form Point Estimators

Given the current shape parameters $\alpha_k,\beta_k$:

$$\widehat\theta_{\text{Bayes}}^{(k)} = \frac{\alpha_k}{\alpha_k+\beta_k}, \qquad\qquad \widehat\theta_{\text{MAP}}^{(k)} = \frac{\alpha_k-1}{\alpha_k+\beta_k-2} \;\; (\text{valid when } \alpha_k>1 \text{ and } \beta_k>1).$$

When $\alpha_k\le 1$ or $\beta_k\le 1$ the Beta density has no interior maximum — it is monotone or U-shaped — so the mode sits at a boundary, $\theta=0$ or $\theta=1$, chosen by whichever parameter is smaller.

## Task 6 — 100-Impression Convergence Simulation

In [4]:
import numpy as np
import plotly.graph_objects as go

class BetaBinomialTracker:
    """Maintains a Beta posterior over a click-through rate under sequential Bernoulli data."""

    def __init__(self, alpha0=1.0, beta0=1.0):
        self.alpha = alpha0
        self.beta = beta0

    def observe(self, y):
        self.alpha += y
        self.beta += (1 - y)

    def posterior_mean(self):
        return self.alpha / (self.alpha + self.beta)

    def posterior_mode(self):
        if self.alpha > 1 and self.beta > 1:
            return (self.alpha - 1) / (self.alpha + self.beta - 2)
        return 0.0 if self.alpha <= self.beta else 1.0


rng = np.random.default_rng(11)
true_ctr = 0.35
n_impressions = 100

tracker = BetaBinomialTracker(alpha0=1.0, beta0=1.0)
mean_track = [tracker.posterior_mean()]
mode_track = [tracker.posterior_mode()]

for _ in range(n_impressions):
    click = 1 if rng.uniform() < true_ctr else 0
    tracker.observe(click)
    mean_track.append(tracker.posterior_mean())
    mode_track.append(tracker.posterior_mode())

steps = list(range(n_impressions + 1))

fig = go.Figure()
fig.add_hline(y=true_ctr, line_dash="dash", line_color="crimson",
              annotation_text=f"true CTR = {true_ctr}")
fig.add_trace(go.Scatter(x=steps, y=mean_track, mode="lines",
                          name="Posterior mean (closed form)", line=dict(color="#1f77b4", width=2.5)))
fig.add_trace(go.Scatter(x=steps, y=mode_track, mode="lines",
                          name="MAP estimate (closed form)", line=dict(color="#2ca02c", width=2, dash="dot")))
fig.update_layout(
    title="CTR Estimate Convergence over 100 Impressions",
    xaxis_title="Impression k", yaxis_title="Estimated CTR",
    template="plotly_white", hovermode="x unified"
)
fig.show()

print(f"Final alpha={tracker.alpha}, beta={tracker.beta}")
print("Final posterior mean:", round(mean_track[-1], 4))
print("Final MAP estimate:  ", round(mode_track[-1], 4))

Final alpha=46.0, beta=56.0
Final posterior mean: 0.451
Final MAP estimate:   0.45


**Analysis:** early on, with only a handful of impressions, the estimators swing noticeably since each single click or non-click changes $\alpha_k+\beta_k$ by a large relative amount and the flat Beta(1,1) prior offers little resistance. As $k$ grows toward 100, each new observation makes a proportionally smaller change to $\alpha_k/(\alpha_k+\beta_k)$, so the curve flattens out and both estimators settle near $\theta_{\text{true}}=0.35$. This mirrors the closed-form mean $\frac{\alpha_0+c_k}{(\alpha_0+\beta_0)+k}$: as $k\to\infty$ the fixed prior constants $\alpha_0,\beta_0$ become negligible next to $k$, and the estimate converges to the empirical click rate $c_k/k$ — the data eventually dominates whatever prior belief we started with, regardless of how that prior was chosen.